# 第2章 2-1: DataFrame 入門

## 学習目標

このノートブックで学ぶこと：

- **pandas** とは何か、なぜ事務作業の自動化に向くのか
- **DataFrame** と **Series** の関係（≒ Excel の表と列）
- 1-3 で書いた「辞書のリスト」から DataFrame を作る（**第1章からの橋渡し**）
- データの中身を覗く `.head()` / `.shape` / `.dtypes` / `.info()`
- 列・行の取り出し（`df["列名"]`、`.loc`、`.iloc`）
- 列に対する集計 `.sum()` / `.mean()` / `.describe()` ほか

**ここから実務の主役・pandas に入ります。** 第1章で書いた `for` ループや辞書集計が、ここからは **1〜2 行で書ける** ようになります。

所要時間の目安: 約30分

## pandas とは

**pandas（パンダス）** は、Python で **表形式のデータ** を扱うためのライブラリです。事務職にとっては「**Excel の表を Python から操作するための道具**」と思ってもらえれば、まず正解です。

### イメージ

```
Excel のシート       ←→   DataFrame（df と略すのが慣習）
Excel の1列          ←→   Series
Excel の見出し行     ←→   columns（列名）
Excel の行番号       ←→   index（行ラベル、0 始まり）
Excel のセル         ←→   1つの値
```

### pandas で何が嬉しいのか

VBA で書くと数十行になる処理が、pandas だと **1〜2 行** で書けます。例えば：

| やりたいこと | VBA の感覚 | pandas |
|---|---|---|
| Excel 読込 | `Workbooks.Open` + ループでセル取得 | `pd.read_excel("path.xlsx")` |
| 合計 | `For` で 1 セルずつ足す | `df["sales"].sum()` |
| 絞り込み | `If ... Then` でループ | `df[df["sales"] >= 10000]` |
| 部署別集計 | 辞書 (Scripting.Dictionary) + ループ | `df.groupby("dept").sum()` |
| Excel 書出 | セル代入 + Format | `df.to_excel("out.xlsx")` |

この章では、まず **DataFrame の作り方と覗き方** から始めます。

## pandas を使う準備

pandas は Colab に最初から入っているので、**インストール不要** です。使うときは毎回 `import` するだけです。

**`pd` と短く書く** のが世界的な慣習で、ほぼ全てのコード例で `pd.read_excel(...)` のように書かれます。

In [ ]:
import pandas as pd

# バージョン確認（Colab はそこそこ新しい版が入っています）
print(pd.__version__)

## DataFrame を作る — 1-3 の「辞書のリスト」がそのまま使える

1-3 で出てきた「辞書のリスト」を覚えていますか？ あれを **そのまま `pd.DataFrame()` に渡す** だけで、pandas の表になります。

つまり 1-3 で学んだデータ構造は、**pandas に渡すための準備運動だった** わけです。

In [ ]:
# 1-3 で書いた形そのまま
employees = [
    {"id": 1, "name": "田中", "department": "営業", "salary": 350000},
    {"id": 2, "name": "鈴木", "department": "開発", "salary": 420000},
    {"id": 3, "name": "佐藤", "department": "営業", "salary": 380000},
    {"id": 4, "name": "山田", "department": "開発", "salary": 450000},
    {"id": 5, "name": "高橋", "department": "総務", "salary": 320000},
]

# DataFrame に変換
df = pd.DataFrame(employees)

# 表として表示される（Colab はキレイに罫線付きで描いてくれます）
df

**ポイント:**

- Colab のセルの **最後の行** に `df` とだけ書くと、**表として整形表示** されます（`print(df)` だと文字だけのシンプル表示）
- 左端の **0, 1, 2... が index**（行ラベル）。Excel の行番号に相当しますが **0 始まり** です
- 1 行目の **id, name, department, salary が columns**（列名）

## 別の作り方: 列ごとの辞書

「辞書のリスト」とは別に、**「列名 → 値のリスト」の辞書** からも作れます。データ量が少ないときの手書き定義によく使います。

In [ ]:
df2 = pd.DataFrame({
    "商品": ["りんご", "ばなな", "みかん", "メロン"],
    "単価": [150, 100, 80, 2800],
    "在庫": [25, 8, 3, 15],
})

df2

## データの中身を覗く（最初に必ず使う5つ）

実際の業務では、Excel から読み込んだデータが **何千行・何万行** になることもあります。最初に必ず使うのが **「ざっと見る」「形を確認する」** ための以下5つです。

| メソッド/属性 | 何が分かる |
|---|---|
| `df.head()` | 先頭5行を表示（行数指定可: `df.head(10)`）|
| `df.tail()` | 末尾5行を表示 |
| `df.shape` | (行数, 列数) のタプル |
| `df.columns` | 列名の一覧 |
| `df.dtypes` | 各列の型（数値 / 文字列など）|
| `df.info()` | 上の情報まとめ + 欠損数 |
| `df.describe()` | 数値列だけの基本統計（件数 / 平均 / 標準偏差 / 最大 / 最小）|

In [ ]:
# 先頭5行（行数指定もできる）
df.head()


In [ ]:
# 形（行数, 列数）
print("shape:", df.shape)        # → (5, 4)
print("列名:", df.columns.tolist())
print("型:")
print(df.dtypes)

In [ ]:
# 一気にまとめて見る
df.info()

In [ ]:
# 数値列の基本統計（Excel の集計表を一発で出すイメージ）
df.describe()

## 列を取り出す（Series が出てくる）

DataFrame から **1列だけ取り出す** と、それは **Series** という型になります。Excel の1列だけを抜き出したイメージで、リストに似ていますが「**index 付きの1列分のデータ**」です。

Series に対しても `.sum()` `.mean()` `.max()` などが直接使えます。

In [ ]:
# 1列だけ取り出す → Series
salaries = df["salary"]
print(type(salaries))    # → <class 'pandas.core.series.Series'>
print(salaries)

print("---")

# Series には集計メソッドがそのまま使える
print("合計:", df["salary"].sum())
print("平均:", df["salary"].mean())
print("最大:", df["salary"].max())
print("最小:", df["salary"].min())
print("件数:", df["salary"].count())

### 複数列をまとめて取り出すには **二重カッコ** `df[[...]]`

1列だけだと `df["name"]`、2列以上だと `df[["name", "salary"]]` のように **リストで包む** のがルールです。

In [ ]:
# 1列 → Series
df["name"]


In [ ]:
# 複数列 → DataFrame（表のまま返る）
df[["name", "salary"]]

## 行を取り出す — `.loc` と `.iloc`

行を取り出すときは2つの書き方があります。**最初は混乱しがち** ですが、こう覚えてください：

- `.loc[ラベル]` — **index ラベルで指定**（Excel の「A列の値」で探すイメージ）
- `.iloc[番号]` — **位置（0 始まりの何番目か）で指定**（Excel の「上から何行目」で探すイメージ）

今は index が `0, 1, 2...` の連番なので、両者は同じ動きに見えますが、後の章で **「日付を index にする」** ような場面で違いが効いてきます。

In [ ]:
# 1 行だけ取り出す
print(df.loc[0])     # index ラベル 0 の行
print("---")
print(df.iloc[0])    # 位置 0（先頭）の行

print("---")

# 範囲指定
print("loc[1:3]（1〜3 を含む = 3行）")
print(df.loc[1:3])

print("---")

print("iloc[1:3]（1〜3未満 = 2行、Python のスライス流）")
print(df.iloc[1:3])

> 💡 `.loc` と `.iloc` の地味だけど重要な違い: **`.loc[1:3]` は終了の 3 を含む**、**`.iloc[1:3]` は 3 を含まない**（1-2 のスライスや `range` と同じ流儀）。最初は戸惑いますが、慣れます。

In [ ]:
# 行と列を同時に指定: df.loc[行, 列]
print(df.loc[0, "name"])              # → 田中（0 行目の name）

# 複数行 × 複数列
print(df.loc[0:2, ["name", "salary"]])

## ちょっと実務寄り: pandas の威力をチラ見せ

1-4 で「**部署別の売上集計**」を `for` + 辞書 で書きましたが、pandas なら **1 行** で同じことが書けます。次節 2-3 でじっくり扱いますが、まずは「**こんなに短く書ける**」という驚きを感じてください。

In [ ]:
# 部署別の給与合計（次節 2-3 で詳しく扱う groupby）
df.groupby("department")["salary"].sum()

In [ ]:
# 部署別の給与平均
df.groupby("department")["salary"].mean()

1-4 で書いた

```python
totals = {}
for d in deals:
    totals[d["dept"]] = totals.get(d["dept"], 0) + d["amount"]
```

が pandas だと `df.groupby("dept")["amount"].sum()` の **1 行** になる、というのが pandas の威力です。次節以降でしっかり使えるようになります。

## 演習

実務感のある問題を3つ用意しました。

### 問題1: DataFrame を作る

下記の商品データから DataFrame を作って表示してください。

```python
products = [
    {"商品": "りんご", "単価": 150, "数量": 30},
    {"商品": "ばなな", "単価": 100, "数量": 50},
    {"商品": "みかん", "単価": 80, "数量": 80},
    {"商品": "メロン", "単価": 2800, "数量": 5},
    {"商品": "ぶどう", "単価": 500, "数量": 12},
]
```

その上で、以下を表示してください:

- `shape`（行数, 列数）
- 列名の一覧
- `describe()` で数値列の基本統計

### 問題2: 列の取り出しと集計

問題1 の DataFrame に対して、以下を求めてください:

- **単価の平均**
- **数量の合計**
- **商品名だけの一覧**（Series で OK）
- **商品名と単価の2列** からなる DataFrame

### 問題3: 特定の行を取り出す

問題1 の DataFrame に対して:

1. **3行目（index = 2）の行全部** を `.loc` で取り出して表示
2. **最後の行** を `.iloc[-1]` で取り出して表示
3. **0〜2行目の「商品」と「単価」だけ** を `.loc` で取り出して表示

ヒント: `.loc[0:2, ["商品", "単価"]]` の形を思い出してください。

In [ ]:
import pandas as pd

products = [
    {"商品": "りんご", "単価": 150, "数量": 30},
    {"商品": "ばなな", "単価": 100, "数量": 50},
    {"商品": "みかん", "単価": 80, "数量": 80},
    {"商品": "メロン", "単価": 2800, "数量": 5},
    {"商品": "ぶどう", "単価": 500, "数量": 12},
]

# 問題1: DataFrame を作る + 基本情報
# ここにコードを書いてください


# 問題2: 列の取り出しと集計
# ここにコードを書いてください


# 問題3: 特定の行を取り出す
# ここにコードを書いてください


<details>
<summary><b>▶ 解答を見る</b></summary>

### 問題1: DataFrame を作る + 基本情報

```python
df = pd.DataFrame(products)
print(df)

print("shape:", df.shape)
print("列名:", df.columns.tolist())
print(df.describe())
```

### 問題2: 列の取り出しと集計

```python
print("単価の平均:", df["単価"].mean())
print("数量の合計:", df["数量"].sum())

print(df["商品"])             # Series（1列）
print(df[["商品", "単価"]])    # DataFrame（2列）
```

### 問題3: 特定の行を取り出す

```python
# 1. 3行目（index=2）の行全部
print(df.loc[2])

# 2. 最後の行（位置で指定）
print(df.iloc[-1])

# 3. 0〜2行目の「商品」と「単価」だけ
print(df.loc[0:2, ["商品", "単価"]])
```

**ポイント:**

- **辞書のリスト → `pd.DataFrame()`** が一番素直な作り方（1-3 で習った形がそのまま使える）
- 1列だけ取り出すと **Series**、複数列 (`[[...]]`) だと **DataFrame** が返る
- 行の取り出しは **`.loc`（ラベル）** と **`.iloc`（位置）** の2系統。今は同じに見えても、後で日付 index などを使うと違いが効く
- `.loc[0:2]` は **2 を含む**、`.iloc[0:2]` は **含まない** — ここは慣れるまで戸惑う場所

</details>

## よくあるエラーと対処

### `KeyError: 'xxx'`

```python
df["sallary"]   # ❌ 列名が typo
```

→ 列名のスペルミス。`df.columns` で正確な列名を確認してください。**全角スペースが混ざっている** こともあるので、Excel から読んだ後は要注意。

### `df["name", "salary"]` で `KeyError`

```python
df["name", "salary"]    # ❌ カッコが1重
df[["name", "salary"]]  # ✅ 2重カッコ（リストで包む）
```

→ 複数列を取り出すときは **二重カッコ**。中身が「列名のリスト」になります。

### `df.loc[1:3]` と `df.iloc[1:3]` で結果の行数が違う

→ バグではなく仕様です。`.loc` はラベル指定なので **終了を含む**、`.iloc` は Python のスライス流で **終了を含まない**。

### `df.sum` と書いたら関数が表示されただけ

```python
df["salary"].sum    # ❌ () がない
df["salary"].sum()  # ✅
```

→ メソッドは呼び出すときに **`()` が必須**。1-5 と同じです。

### Excel の数字列がなぜか文字列になっている

→ pandas は **空欄や全角数字、`-` などが混ざっている列を文字列とみなす** ことがあります。`df.dtypes` で確認して、必要なら `pd.to_numeric(df["列名"], errors="coerce")` で数値に変換します。これは 2-4 「データ加工」で詳しく扱います。

## まとめ

このノートブックで学んだこと：

- **pandas** は表形式データの道具、Excel のシート ≒ **DataFrame**、列 ≒ **Series**
- **`import pandas as pd`** で読み込む（Colab は標準搭載）
- **辞書のリスト → `pd.DataFrame()`** で1-3 から直接つながる
- 中身確認の5点セット: **`head` / `shape` / `columns` / `dtypes` / `info` / `describe`**
- 列取り出し: **`df["列"]`**（Series）、**`df[["列1", "列2"]]`**（DataFrame、二重カッコ）
- 行取り出し: **`.loc[ラベル]`**（終了を含む）、**`.iloc[位置]`**（終了を含まない）
- Series に対して **`.sum()` `.mean()` `.max()` `.min()`** が直接使える
- **`groupby`** で部署別集計が **1行** で書ける（次節以降の伏線）

次のノートブック `02-2_read_excel_csv.ipynb` では、**実際の Excel/CSV ファイルを読み込む** 方法を学びます。Drive 上のファイルを `pd.read_excel("...")` で読み込み、ここで覚えた DataFrame の使い方が本領を発揮します。